In [4]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
  work_dir = '/content/drive/MyDrive/COMP720Project/SampledExplanation'
except:
  IN_COLAB = False
  work_dir = input()

Mounted at /content/drive


In [5]:
# !mkdir /content/drive/MyDrive/COMP720Project/SampledExplanation
# !cp /content/drive/MyDrive/COMP720Project/hbf/explanation_res.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_hbf.csv
# !cp /content/drive/MyDrive/COMP720Project/cf/temp/explanations.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_cf.csv
# !cp /content/drive/MyDrive/COMP720Project/rcbf_all_features/temp/explanations.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_cbf.csv

In [6]:
import os
os.chdir(work_dir)
os.listdir(work_dir)

['explanations_cf.csv',
 'explanations_cbf.csv',
 'explanations_hbf.csv',
 'summary_means.csv',
 'scored_cbf.csv',
 'score_distributions.csv',
 'scored_cf.csv',
 'scored_hbf.csv']

In [ ]:
import re
import numpy as np
import pandas as pd

SAMPLE_PERS = 100
RANDOM_SEED = 42


In [21]:
pd.set_option('max_colwidth', 512)
pd.set_option('display.max_rows', 500)


In [8]:
FILES = {
    "cbf": ("explanations_cbf.csv", "Content-Based"),
    "cf": ("explanations_cf.csv", "Collaborative"),
    "hbf": ("explanations_hbf.csv", "Hybrid"),
}

In [9]:
################
### Patterns ###
################

PAT_WATCH = re.compile(
    r'Because you watched "(?P<src>.*?)", which (?P<clauses>.*?), '
    r'we think you\'ll like "(?P<rec>.*?)"\.'
)
PAT_NO_HISTORY = re.compile(
    r'"(?P<rec>.*?)" is recommended based on your overall viewing patterns\.'
)
PAT_CF_ITEMS = re.compile(
    r'Because you enjoyed (?P<items>.*?), we think you\'ll like "(?P<rec>.*?)"\.'
)
PAT_CF_GENERIC = re.compile(
    r'"(?P<rec>.*?)" is broadly similar to your recent viewing\.'
)
ITEM_PAT = re.compile(r'(?P<title>.*?) \(your rating: (?P<rating>[\d.]+)\)')

# PAT_WATCH captures the whole "which ... ," middle clause as one blob, since
# the generator (explanations.py) joins its reason clauses with " and " in
# whatever order/subset fired (genre isn't always first, and isn't guaranteed
# to be present at all) rather than a fixed sequence of optional groups.
# Splitting that blob on " and " and matching each piece below is what lets
# parse_cbf_hbf handle any combination.
CLAUSE_GENRE = re.compile(r'shares the (?P<genres>[^()]*) genre\(s\)')
CLAUSE_THEME = "has a similar theme/plot"
CLAUSE_STORY = "touches on similar story elements"
CLAUSE_TASTE = "is often watched by users with similar taste to yours"
CLAUSE_STYLE = "is broadly similar in style"

In [10]:
def parse_cbf_hbf(exp):
    m = PAT_WATCH.match(exp)
    if m:
        src, rec = m.group("src").strip(), m.group("rec").strip()
        clauses = [c.strip() for c in m.group("clauses").split(" and ")]

        genres = []
        theme = story = taste = False
        recognized = 0
        for clause in clauses:
            gm = CLAUSE_GENRE.fullmatch(clause)
            if gm:
                genres = [g.strip() for g in gm.group("genres").split(",") if g.strip()]
                recognized += 1
            elif clause == CLAUSE_THEME:
                theme = True
                recognized += 1
            elif clause == CLAUSE_STORY:
                story = True
                recognized += 1
            elif clause == CLAUSE_TASTE:
                taste = True
                recognized += 1
            # else: unrecognized clause text (e.g. the CLAUSE_STYLE fallback,
            # which only ever appears alone) -- falls through to style_generic below.

        if genres:
            template = "genre"
        elif recognized == 0:
            # No genre and nothing else recognized either -- covers CLAUSE_STYLE
            # ("is broadly similar in style") and any unrecognized clause text.
            template = "style_generic"
        else:
            template = "+".join(name for name, present in
                                 (("theme", theme), ("story", story), ("taste", taste)) if present)

        return {"template": template, "src": src, "rec": rec,
                "n_genres": len(genres), "theme": theme, "story": story, "taste": taste}

    m = PAT_NO_HISTORY.match(exp)
    if m:
        return {"template": "no_history", "src": None, "rec": m.group("rec").strip(),
                "n_genres": 0, "theme": False, "story": False, "taste": False}

    return {"template": "unparsed", "src": None, "rec": None,
            "n_genres": 0, "theme": False, "story": False, "taste": False}



In [11]:
def parse_cf(exp):
    m = PAT_CF_ITEMS.match(exp)
    if m:
        items_str = m.group("items")
        parts = re.split(r"(?<=\)), ", items_str)
        titles, ratings = [], []
        for p in parts:
            im = ITEM_PAT.match(p.strip())
            if im:
                titles.append(im.group("title").strip())
                ratings.append(float(im.group("rating")))
        return {"template": "items", "rec": m.group("rec").strip(), "titles": titles,
                "ratings": ratings, "n_items": len(titles)}
    m = PAT_CF_GENERIC.match(exp)
    if m:
        return {"template": "generic", "rec": m.group("rec").strip(), "titles": [], "ratings": [], "n_items": 0}
    return {"template": "unparsed", "rec": None, "titles": [], "ratings": [], "n_items": 0}




In [12]:

results = {}

for key, (fname, label) in FILES.items():
    df = pd.read_csv(fname)

    rows = []
    for _, row in df.iterrows():
        exp = row["explanation"]
        if key == "cf":
            feat = parse_cf(exp)
            n_evidence = feat["n_items"]
            self_rec = feat["rec"] is not None and feat["rec"].strip().lower() in [t.lower() for t in feat["titles"]]
        else:
            feat = parse_cbf_hbf(exp)
            n_evidence = feat["n_genres"]
            self_rec = (feat["src"] is not None and feat["rec"] is not None
                        and feat["src"].strip().lower() == feat["rec"].strip().lower())

        rows.append({
            "user_id": row["user_id"], "item_id": row["item_id"], "explanation": exp,
            "template": feat["template"], "n_evidence": n_evidence, "self_rec_bug": self_rec,
        })
    parsed = pd.DataFrame(rows)
    parsed.to_csv(f"scored_{key}.csv", index=False)
    results[key] = (label, parsed)

# ---- Parsing report ----
print("=" * 70)
print("Parsed the full explanation file per method (no sampling)")
print("=" * 70)

Parsed the full explanation file per method (no sampling)


In [13]:
summary_rows = []
for key, (label, parsed) in results.items():
    self_rec_rate = parsed["self_rec_bug"].mean()
    template_dist = parsed["template"].value_counts(normalize=True).round(3).to_dict()
    unparsed_count = int((parsed["template"] == "unparsed").sum())
    print(f"\n--- {label} ---")
    print(f"  Unparsed count: {unparsed_count} / {len(parsed)}")
    print(f"  Self-recommendation bug rate: {self_rec_rate:.1%}")
    print(f"  Template mix: {template_dist}")
    summary_rows.append({
        "method": label, "file": key,
        "unparsed_count": unparsed_count,
        "self_rec_bug_rate": round(self_rec_rate, 3),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("summary_means.csv", index=False)
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
display(summary_df)


--- Content-Based ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'genre': 0.974, 'style_generic': 0.019, 'story': 0.007}

--- Collaborative ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'items': 0.553, 'generic': 0.447}

--- Hybrid ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'genre': 0.945, 'taste': 0.042, 'style_generic': 0.007, 'story': 0.003, 'story+taste': 0.002, 'theme+taste': 0.001}

SUMMARY TABLE


,method,file,unparsed_count,self_rec_bug_rate
0,Content-Based,cbf,0,0.0
1,Collaborative,cf,0,0.0
2,Hybrid,hbf,0,0.0


## LLM plausibility judge (sample 50 items)

In [15]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.0 MB/s eta 0:00:0000:010:01


In [ ]:
import os
import time

import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

client = anthropic.Anthropic()

JUDGE_MODEL = "claude-sonnet-5"

JUDGE_PROMPT = """You will be shown one movie recommendation explanation at a time. Each states a reason for recommending a film based on something the user previously watched or rated.
Using your knowledge of the films named, judge whether the stated reason is credible: would a viewer who knows these films find this a sensible basis for the recommendation, or would the connection seem arbitrary?
Rate plausibility from 1 to 5, where 1 means the cited films have no meaningful relationship to the recommendation and 5 means the connection is clear and well-founded. Judge only what the sentence actually argues -- do not credit a recommendation that happens to be good if the stated reason does not support it.
Give the rating and one sentence of justification naming the specific films. Do not consider how much evidence is cited; a reason citing one film may be more credible than one citing three.
You have a web_search tool. Use it to confirm plot, genre, or thematic details of the named films whenever you are not fully confident from memory alone -- the judgment should reflect what the films are actually about, not a guess.
Respond with nothing but the rating and justification, in exactly this format:
Rating: <integer 1-5>
Justification: <one sentence, naming the specific films>"""

LLM_SAMPLE_N = 50   # rows per file
LLM_RNG_SEED = 42

In [17]:
RATING_PAT = re.compile(r"Rating:\s*([1-5]).*?Justification:\s*(.+)", re.DOTALL)

def judge_plausibility(explanation):
    try:
        response = client.messages.create(
            model=JUDGE_MODEL,
            max_tokens=8192,
            system=JUDGE_PROMPT,
            output_config={"effort": "medium"},
            tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 3}],
            messages=[{"role": "user", "content": explanation}],
        )
    except anthropic.APIError as e:
        return {"rating": None, "justification": f"API error: {e}"}

    text = " ".join(b.text for b in response.content if b.type == "text").strip()
    m = RATING_PAT.search(text)
    if m:
        return {"rating": int(m.group(1)), "justification": m.group(2).strip()}
    return {"rating": None, "justification": text or None}

In [ ]:
llm_rows = []

for key, (label, parsed) in results.items():
    sample = parsed.sample(n=min(LLM_SAMPLE_N, len(parsed)), random_state=LLM_RNG_SEED)
    for i, (_, row) in enumerate(sample.iterrows()):
        judged = judge_plausibility(row["explanation"])
        llm_rows.append({
            "method": label, "file": key,
            "user_id": row["user_id"], "item_id": row["item_id"],
            "explanation": row["explanation"], "template": row["template"],
            "llm_plausibility": judged["rating"], "llm_justification": judged["justification"],
        })
        print(f"[{label}] {i + 1}/{len(sample)}  rating={judged['rating']}")

llm_df = pd.DataFrame(llm_rows)
llm_df.to_csv("llm_plausibility_sample.csv", index=False)

[Content-Based] 1/50  rating=3
[Content-Based] 2/50  rating=3
[Content-Based] 3/50  rating=3
[Content-Based] 4/50  rating=5
[Content-Based] 5/50  rating=5
[Content-Based] 6/50  rating=4
[Content-Based] 7/50  rating=4
[Content-Based] 8/50  rating=5
[Content-Based] 9/50  rating=2
[Content-Based] 10/50  rating=4
[Content-Based] 11/50  rating=4
[Content-Based] 12/50  rating=3
[Content-Based] 13/50  rating=2
[Content-Based] 14/50  rating=4
[Content-Based] 15/50  rating=3
[Content-Based] 16/50  rating=4
[Content-Based] 17/50  rating=2
[Content-Based] 18/50  rating=5
[Content-Based] 19/50  rating=2
[Content-Based] 20/50  rating=4
[Content-Based] 21/50  rating=3
[Content-Based] 22/50  rating=3
[Content-Based] 23/50  rating=4
[Content-Based] 24/50  rating=3
[Content-Based] 25/50  rating=1
[Content-Based] 26/50  rating=2
[Content-Based] 27/50  rating=4
[Content-Based] 28/50  rating=4
[Content-Based] 29/50  rating=3
[Content-Based] 30/50  rating=2
[Content-Based] 31/50  rating=5
[Content-Based] 3

,method,file,user_id,item_id,explanation,template,llm_plausibility,llm_justification
0,Content-Based,cbf,26,875,"Because you watched ""Harry Potter and the Deat...",genre,3,"Both ""Harry Potter and the Deathly Hallows: Pa..."
1,Content-Based,cbf,36,427,"Because you watched ""A Beautiful Mind"", which ...",genre,3,"Both ""A Beautiful Mind"" and ""To Kill a Mocking..."
2,Content-Based,cbf,37,7835,"Because you watched ""Iron Man"", which shares t...",genre,3,"""Iron Man"" and ""Inception"" both fall under Act..."
3,Content-Based,cbf,33,86,"Because you watched ""Howl's Moving Castle"", wh...",genre,5,"Both ""Howl's Moving Castle"" and ""Princess Mono..."
4,Content-Based,cbf,20,4122,"Because you watched ""A Nightmare on Elm Street...",genre,5,"""A Nightmare on Elm Street"" and ""Wes Craven's ..."
...,...,...,...,...,...,...,...,...
145,Hybrid,hbf,13,3691,"Because you watched ""The Ladies Man"", which sh...",genre,2,"""The Ladies Man"" is a comedy about a womanizin..."
146,Hybrid,hbf,46,401,"Because you watched ""Chinatown"", which shares ...",genre,4,"""Chinatown"" and ""Rear Window"" are both acclaim..."
147,Hybrid,hbf,30,14085,"Because you watched ""Lady Bird"", which shares ...",genre,3,"""Lady Bird"" and ""Isle of Dogs"" are both direct..."
148,Hybrid,hbf,21,15141,"Because you watched ""Close"", which shares the ...",genre,2,"""Close"" is a 2019 British action thriller film..."


In [22]:
llm_df

,method,file,user_id,item_id,explanation,template,llm_plausibility,llm_justification
0,Content-Based,cbf,26,875,"Because you watched ""Harry Potter and the Deathly Hallows: Part 2"", which shares the Fantasy genre(s), we think you'll like ""Hellboy"".",genre,3,"Both ""Harry Potter and the Deathly Hallows: Part 2"" and ""Hellboy"" fall under the broad Fantasy genre with supernatural/magical elements and epic good-vs-evil battles, but their tone, setting (British boarding-school wizarding world vs. dark comic-book horror-action), and audience skew quite differently, making the shared-genre link plausible but fairly generic."
1,Content-Based,cbf,36,427,"Because you watched ""A Beautiful Mind"", which shares the Drama genre(s), we think you'll like ""To Kill a Mockingbird"".",genre,3,"Both ""A Beautiful Mind"" and ""To Kill a Mockingbird"" are acclaimed dramas built around a principled protagonist facing hostility (mental illness stigma vs. racial injustice), so the shared genre link is real but fairly generic and doesn't capture any deeper thematic connection beyond both being prestige dramas."
2,Content-Based,cbf,37,7835,"Because you watched ""Iron Man"", which shares the Action, Adventure, Science Fiction genre(s), we think you'll like ""Inception"".",genre,3,"""Iron Man"" and ""Inception"" both fall under Action/Adventure/Sci-Fi, but beyond broad genre overlap they differ sharply in tone, style, and narrative focus (superhero origin story vs. heist/dream-thriller), making the connection genre-technical rather than substantively meaningful."
3,Content-Based,cbf,33,86,"Because you watched ""Howl's Moving Castle"", which shares the Adventure, Animation, Fantasy genre(s) and touches on similar story elements, we think you'll like ""Princess Mononoke"".",genre,5,"Both ""Howl's Moving Castle"" and ""Princess Mononoke"" are Studio Ghibli fantasy adventures directed by Hayao Miyazaki that blend animation, magical transformation themes, and environmental/anti-war messaging, making the genre and thematic overlap well-founded."
4,Content-Based,cbf,20,4122,"Because you watched ""A Nightmare on Elm Street"", which shares the Horror genre(s) and touches on similar story elements, we think you'll like ""Wes Craven's New Nightmare"".",genre,5,"""A Nightmare on Elm Street"" and ""Wes Craven's New Nightmare"" are both directed by Wes Craven, feature Freddy Krueger and Nancy (Heather Langenkamp), and share horror/slasher themes, with the sequel directly building on the original's story and meta-mythology."
5,Content-Based,cbf,33,82,"Because you watched ""Howl's Moving Castle"", which shares the Adventure, Fantasy genre(s), we think you'll like ""The Lord of the Rings: The Return of the King"".",genre,4,"""Howl's Moving Castle"" and ""The Lord of the Rings: The Return of the King"" both belong squarely to the Adventure/Fantasy genre with epic quests, magic, and war themes, making genre-based similarity reasonable, though the films differ significantly in tone, animation style, and target audience."
6,Content-Based,cbf,31,511,"Because you watched ""Leaving Las Vegas"", which shares the Drama genre(s), we think you'll like ""Dead Man Walking"".",genre,4,"""Leaving Las Vegas"" and ""Dead Man Walking"" are both mid-90s serious dramas with heavyweight lead performances dealing with mortality and moral/emotional despair, so the genre link is real, though the connection is fairly generic since ""Drama"" alone covers a huge range of otherwise dissimilar films."
7,Content-Based,cbf,25,68,"Because you watched ""Reservoir Dogs"", which shares the Crime genre(s) and has a similar theme/plot, we think you'll like ""Snatch"".",genre,5,"""Reservoir Dogs"" and ""Snatch"" are both stylish, non-linear crime films featuring ensembles of criminals, heists gone wrong, dark humor, and violent underworld dealings, making the genre and thematic connection well-founded."
8,Content-Based,cbf,42,1217,"Because you watched ""Menace II Society"", which shares the Crime, Thril

In [23]:
print("=" * 70)
print(f"LLM plausibility sample: {LLM_SAMPLE_N} rows/file (seed={LLM_RNG_SEED})")
print("=" * 70)

for method, group in llm_df.groupby("method"):
    rated = group["llm_plausibility"].dropna()
    print(f"\n--- {method} ---")
    print(f"  Rated: {len(rated)} / {len(group)}")
    if len(rated):
        print(f"  Mean LLM plausibility: {rated.mean():.2f}")

display(llm_df[["method", "user_id", "item_id", "template", "llm_plausibility", "llm_justification"]])

LLM plausibility sample: 50 rows/file (seed=42)

--- Collaborative ---
  Rated: 50 / 50
  Mean LLM plausibility: 1.68

--- Content-Based ---
  Rated: 50 / 50
  Mean LLM plausibility: 3.30

--- Hybrid ---
  Rated: 50 / 50
  Mean LLM plausibility: 2.98


,method,user_id,item_id,template,llm_plausibility,llm_justification
0,Content-Based,26,875,genre,3,"Both ""Harry Potter and the Deathly Hallows: Part 2"" and ""Hellboy"" fall under the broad Fantasy genre with supernatural/magical elements and epic good-vs-evil battles, but their tone, setting (British boarding-school wizarding world vs. dark comic-book horror-action), and audience skew quite differently, making the shared-genre link plausible but fairly generic."
1,Content-Based,36,427,genre,3,"Both ""A Beautiful Mind"" and ""To Kill a Mockingbird"" are acclaimed dramas built around a principled protagonist facing hostility (mental illness stigma vs. racial injustice), so the shared genre link is real but fairly generic and doesn't capture any deeper thematic connection beyond both being prestige dramas."
2,Content-Based,37,7835,genre,3,"""Iron Man"" and ""Inception"" both fall under Action/Adventure/Sci-Fi, but beyond broad genre overlap they differ sharply in tone, style, and narrative focus (superhero origin story vs. heist/dream-thriller), making the connection genre-technical rather than substantively meaningful."
3,Content-Based,33,86,genre,5,"Both ""Howl's Moving Castle"" and ""Princess Mononoke"" are Studio Ghibli fantasy adventures directed by Hayao Miyazaki that blend animation, magical transformation themes, and environmental/anti-war messaging, making the genre and thematic overlap well-founded."
4,Content-Based,20,4122,genre,5,"""A Nightmare on Elm Street"" and ""Wes Craven's New Nightmare"" are both directed by Wes Craven, feature Freddy Krueger and Nancy (Heather Langenkamp), and share horror/slasher themes, with the sequel directly building on the original's story and meta-mythology."
5,Content-Based,33,82,genre,4,"""Howl's Moving Castle"" and ""The Lord of the Rings: The Return of the King"" both belong squarely to the Adventure/Fantasy genre with epic quests, magic, and war themes, making genre-based similarity reasonable, though the films differ significantly in tone, animation style, and target audience."
6,Content-Based,31,511,genre,4,"""Leaving Las Vegas"" and ""Dead Man Walking"" are both mid-90s serious dramas with heavyweight lead performances dealing with mortality and moral/emotional despair, so the genre link is real, though the connection is fairly generic since ""Drama"" alone covers a huge range of otherwise dissimilar films."
7,Content-Based,25,68,genre,5,"""Reservoir Dogs"" and ""Snatch"" are both stylish, non-linear crime films featuring ensembles of criminals, heists gone wrong, dark humor, and violent underworld dealings, making the genre and thematic connection well-founded."
8,Content-Based,42,1217,genre,2,"""Menace II Society"" is a gritty urban crime drama about gang life and violence in South Central LA, while ""Nick of Time"" is a real-time political assassination thriller centered on a coerced accountant, so beyond both being loosely ""crime/thriller"" the actual story elements and settings share little in common."
9,Content-Based,6,3492,genre,4,"""Sleepless in Seattle"" and ""It Could Happen to You"" are both mid-1990s romantic comedy-dramas built around a chance-of-fate premise and gentle sentimental tone, so the shared Comedy/Drama/Romance genre link is a sensible, if fairly generic, basis for the recommendation."
